# Zigrupt benchmark analysis

This notebook reads **one explicitly selected result directory**. Set `DATA_DIR` below to the result directory you want to analyze. The template contains no embedded results. Run the cells in order. Empty and check-only results still produce a coverage report.

**What is measured.** Saturated throughput counts *unique events per second*. A broadcast event reaches every required consumer, so `deliveries_per_second` can be higher. Latency starts at scheduled arrival and ends at the consumer clock read; it includes producer backpressure and queueing. The six offered loads are fractions of the slowest selected implementation's median saturated throughput for the same workload (Zigrupt's own median in standalone mode). These results describe this machine, placement, and run; compare implementations only where each has valid measurements.


In [ ]:
from __future__ import annotations

import csv
import math
import statistics
from collections import Counter, defaultdict
from collections.abc import Hashable
from dataclasses import dataclass
from html import escape
from pathlib import Path
from typing import Callable, Iterable, TypeVar

import matplotlib.pyplot as plt
from IPython.display import HTML, Markdown, display
from matplotlib.lines import Line2D

# Set this path explicitly when the CSVs are elsewhere. Point it at a result directory
# containing throughput.csv and, for measured runs, latency.csv.
DATA_DIR = Path("results/your-run-directory").resolve()

COLORS = {"zigrupt": "#005A91", "java": "#B94A00", "rust": "#007A57"}
MARKERS = {"zigrupt": "o", "java": "s", "rust": "^"}
plt.rcParams.update(
    {
        "figure.dpi": 110,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.alpha": 0.22,
        "font.size": 10,
    }
)


## 1. Load and account for the run

**Question:** Which workloads can support a measurement comparison, and why are others missing? The table retains correctness failures, blocked cases, and incomplete measurements. A valid latency repeat requires exactly one valid row for every expected consumer; duplicate or missing consumer rows do not become a partial percentile.


In [ ]:
Row = dict[str, str]
Key = tuple[str, str]
Item = TypeVar("Item")
GroupKey = TypeVar("GroupKey", bound=Hashable)


def read_csv(path: Path) -> list[Row]:
    """Read a runner CSV; missing latency is valid for check-only runs."""
    if not path.exists():
        return []
    with path.open(newline="") as stream:
        reader = csv.DictReader(stream)
        return [
            {key: value or "" for key, value in row.items() if key is not None}
            for row in reader
        ]


def number(row: Row, field: str) -> float | None:
    """Parse a finite numeric field, preserving absence as None."""
    try:
        value = float(row.get(field, ""))
    except (TypeError, ValueError):
        return None
    return value if math.isfinite(value) else None


def required_number(row: Row, field: str) -> float:
    """Read a field already validated by the run filter."""
    value = number(row, field)
    if value is None:
        raise ValueError(f"Missing numeric {field} for {row.get('workload', '?')}")
    return value


def integer(row: Row, field: str) -> int | None:
    """Read an integral CSV field when present."""
    value = number(row, field)
    return int(value) if value is not None and value.is_integer() else None


def is_valid(row: Row) -> bool:
    """Use the runner validity flag, independent of mode."""
    return row.get("valid", "").lower() == "true"


def workload_label(row: Row) -> str:
    """Label capacity and payload together."""
    return f"{row.get('capacity', '?')} slots / {row.get('payload', '?')} B"


def topology(row: Row) -> str:
    """Label producer and consumer counts."""
    return f"{row.get('producers', '?')}P → {row.get('consumer_count', '?')}C"


throughput_rows = read_csv(DATA_DIR / "throughput.csv")
latency_rows = read_csv(DATA_DIR / "latency.csv")
if not (DATA_DIR / "throughput.csv").exists():
    raise FileNotFoundError(f"No throughput.csv in {DATA_DIR}; set DATA_DIR above.")
required = {
    "workload",
    "implementation",
    "mode",
    "repeat",
    "valid",
    "producers",
    "consumer_count",
    "capacity",
    "payload",
}
for filename, rows in (
    ("throughput.csv", throughput_rows),
    ("latency.csv", latency_rows),
):
    path = DATA_DIR / filename
    if path.exists():
        with path.open(newline="") as stream:
            missing = required - set(csv.DictReader(stream).fieldnames or [])
        if missing:
            raise ValueError(f"{filename} is missing fields: {sorted(missing)}")
print(f"Selected data directory: {DATA_DIR}")
print(
    f"Loaded {len(throughput_rows)} throughput/check rows and "
    f"{len(latency_rows)} latency consumer/error rows."
)


In [ ]:
@dataclass(frozen=True)
class LatencyRun:
    """A complete latency repeat with one row per consumer."""

    row: Row
    consumers: tuple[Row, ...]

    def worst(self, field: str) -> float:
        return max(required_number(consumer, field) for consumer in self.consumers)


saturation = [
    row
    for row in throughput_rows
    if row.get("mode") == "throughput"
    and is_valid(row)
    and (number(row, "events_per_second") or 0) > 0
]
latency_groups: dict[tuple[str, str, str, str], list[Row]] = defaultdict(list)
for row in latency_rows:
    if row.get("mode") == "latency":
        latency_groups[
            (
                row.get("workload", ""),
                row.get("implementation", ""),
                row.get("repeat", ""),
                row.get("fraction", ""),
            )
        ].append(row)

latency_runs: list[LatencyRun] = []
incomplete_latency: Counter[Key] = Counter()
for group in latency_groups.values():
    first = group[0]
    expected = integer(first, "consumer_count")
    consumers = [integer(row, "consumer") for row in group]
    consumer_ids = [value for value in consumers if value is not None]
    complete = (
        expected is not None
        and expected > 0
        and len(group) == expected
        and len(consumer_ids) == len(consumers)
        and sorted(consumer_ids) == list(range(expected))
        and all(
            is_valid(row)
            and (number(row, "samples") or 0) > 0
            and number(row, "fraction") is not None
            and all(
                number(row, field) is not None
                for field in ("p50_ns", "p99_ns", "p999_ns")
            )
            for row in group
        )
    )
    if complete:
        latency_runs.append(
            LatencyRun(
                first,
                tuple(sorted(group, key=lambda row: integer(row, "consumer") or 0)),
            )
        )
    else:
        incomplete_latency[
            (first.get("workload", ""), first.get("implementation", ""))
        ] += 1


def group_by(
    rows: Iterable[Item], key: Callable[[Item], GroupKey]
) -> dict[GroupKey, list[Item]]:
    """Group records without discarding their concrete type."""
    groups: dict[GroupKey, list[Item]] = defaultdict(list)
    for row in rows:
        groups[key(row)].append(row)
    return groups


implementations = sorted(
    {
        row.get("implementation", "")
        for row in throughput_rows + latency_rows
        if row.get("implementation")
    }
)
for name in implementations:
    if name not in COLORS:
        COLORS[name] = "#777777"
        MARKERS[name] = "D"
all_keys = sorted(
    {
        (row.get("workload", ""), row.get("implementation", ""))
        for row in throughput_rows + latency_rows
        if row.get("workload")
    }
)


def coverage_table() -> None:
    """Show attempts, valid repeats, and known failure reasons."""
    heads = "<tr><th>Workload</th><th>Implementation</th><th>Check</th><th>Saturation</th><th>Latency</th><th>Issue / reason</th></tr>"
    body = []
    for workload, impl in all_keys:
        trows = [
            row
            for row in throughput_rows
            if row.get("workload") == workload and row.get("implementation") == impl
        ]
        lrows = [
            run
            for run in latency_runs
            if (run.row.get("workload"), run.row.get("implementation"))
            == (workload, impl)
        ]
        checks = [row for row in trows if row.get("mode") == "check"]
        attempts = [row for row in trows if row.get("mode") == "throughput"]
        raw_l = [
            group
            for group in latency_groups.values()
            if (group[0].get("workload"), group[0].get("implementation"))
            == (workload, impl)
        ]
        issues = list(
            dict.fromkeys(
                (row.get("reason") or row.get("error") or "blocked")
                for row in trows
                if row.get("mode") == "blocked"
            )
        )
        issues += list(
            dict.fromkeys(
                (row.get("error") or "invalid check/measurement")
                for row in trows + [g[0] for g in raw_l]
                if row.get("mode") != "blocked" and not is_valid(row)
            )
        )
        if incomplete_latency[(workload, impl)]:
            issues.append(
                f"{incomplete_latency[(workload, impl)]} incomplete latency repeat(s)"
            )
        check = f"{sum(map(is_valid, checks))}/{len(checks)}" if checks else "—"
        sat = (
            f"{sum(row in saturation for row in attempts)}/{len(attempts)}"
            if attempts
            else "—"
        )
        lat = f"{len(lrows)}/{len(raw_l)}" if raw_l else "—"
        body.append(
            "<tr>"
            + "".join(
                f"<td>{escape(str(value))}</td>"
                for value in (workload, impl, check, sat, lat, "; ".join(issues) or "—")
            )
            + "</tr>"
        )
    display(
        HTML(
            "<div style='max-height:480px;overflow:auto'><table>"
            + heads
            + "".join(body)
            + "</table></div>"
        )
    )


coverage_table()
blocked_count = sum(row.get("mode") == "blocked" for row in throughput_rows)
invalid_count = sum(
    row.get("mode") != "blocked" and not is_valid(row) for row in throughput_rows
)
invalid_count += sum(not is_valid(group[0]) for group in latency_groups.values())
display(
    Markdown(
        f"**Reading this run:** {len({row['workload'] for row in saturation})} workloads "
        f"have valid saturated throughput; {len(latency_runs)} complete latency repeats "
        f"are available. The CSVs also record {blocked_count} blocked cases and "
        f"{invalid_count} invalid attempts. A dash means that mode was not attempted."
    )
)


## 2. Saturated throughput

**Question:** How many unique events can each workload sustain? Each marker is the median of valid repeats, with horizontal whiskers from the smallest to the largest repeat. Rows pair ring capacity with payload size. Compare a given row across implementations, then inspect the whisker span before drawing conclusions. Missing or blocked rows have no measured marker; their reasons remain in the coverage table.


In [ ]:
def legend_handles() -> list[Line2D]:
    """Use the same implementation encoding in every figure."""
    return [
        Line2D(
            [0], [0], color=COLORS[name], marker=MARKERS[name], label=name,
            lw=2.3, markersize=9, markeredgecolor="#1A1A1A",
            markeredgewidth=0.8,
        )
        for name in implementations
    ]


def horizontal_panels(
    rows: list[Row],
    field: str,
    scale: float,
    unit: str,
    title: str,
    log_axis: bool = False,
) -> None:
    """Show repeat medians and min–max whiskers on topology plots."""
    usable = [row for row in rows if (number(row, field) or 0) > 0]
    if not usable:
        display(Markdown(f"No valid {title.lower()} measurements in this run."))
        return
    values = [required_number(row, field) / scale for row in usable]
    lower, upper = min(values), max(values)
    limits = (lower / 1.7, upper * 1.7) if log_axis else (0, upper * 1.12)
    for topo in sorted({topology(row) for row in usable}):
        selected = [row for row in usable if topology(row) == topo]
        label_by_shape = {
            (
                int(required_number(row, "capacity")),
                int(required_number(row, "payload")),
            ): workload_label(row)
            for row in selected
        }
        labels = [label_by_shape[key] for key in sorted(label_by_shape)]
        fig, ax = plt.subplots(figsize=(10, max(2.8, 0.75 * len(labels) + 1.2)))
        for index, name in enumerate(implementations):
            offset = (index - (len(implementations) - 1) / 2) * 0.20
            for position, label in enumerate(labels):
                samples = [
                    required_number(row, field) / scale
                    for row in selected
                    if row.get("implementation") == name
                    and workload_label(row) == label
                ]
                if not samples:
                    continue
                y = position + offset
                median = statistics.median(samples)
                ax.errorbar(
                    median,
                    y,
                    xerr=[[median - min(samples)], [max(samples) - median]],
                    fmt=MARKERS[name],
                    color=COLORS[name],
                    markersize=10,
                    markeredgecolor="#1A1A1A",
                    markeredgewidth=0.8,
                    capsize=5,
                    capthick=2.3,
                    elinewidth=2.3,
                    zorder=3,
                )
        ax.set(
            yticks=range(len(labels)),
            yticklabels=labels,
            xlim=limits,
            xlabel=unit,
            title=f"{title} · {topo}",
        )
        if log_axis:
            ax.set_xscale("log")
        ax.invert_yaxis()
        ax.grid(axis="y", alpha=0.1)
        ax.legend(handles=legend_handles(), loc="best", ncol=len(implementations))
        fig.text(
            0.5, 0.015, "Marker: median  •  whiskers: minimum–maximum repeat",
            ha="center", fontsize=9,
        )
        fig.tight_layout(rect=(0, 0.055, 1, 1))
        plt.show()


horizontal_panels(
    saturation,
    "events_per_second",
    1e6,
    "Million unique events / second",
    "Saturated throughput",
)
if saturation:
    medians = [
        statistics.median(required_number(row, "events_per_second") for row in group)
        for group in group_by(
            saturation, lambda row: (row["workload"], row["implementation"])
        ).values()
    ]
    display(
        Markdown(
            f"**Readout:** {len(medians)} workload–implementation pairs have "
            f"saturated medians spanning {min(medians) / 1e6:.2f}–"
            f"{max(medians) / 1e6:.2f} million unique events/s. "
            "Broadcast deliveries/s would multiply by consumer count; the plot "
            "keeps unique-event rates comparable."
        )
    )


## 3. Tail latency under shared offered load

**Question:** How does the slowest consumer's tail change as the offered rate rises? For each complete repeat, take the **maximum of the consumer p99 values**. Then plot the median of those repeat values. The shaded band spans valid repeat minima and maxima, where there is more than one repeat. This is a median of per-consumer summaries, **not a pooled p99 of raw events**. The vertical axis is logarithmic and measured in microseconds.


In [ ]:
def workload_panels(
    metric: Callable[[LatencyRun], float | None],
    ylabel: str,
    heading: str,
    log_axis: bool = False,
    reference: float | None = None,
) -> None:
    """Plot repeat medians and ranges, omitting absent metrics."""
    if not latency_runs:
        display(Markdown(f"No complete latency repeats for {heading.lower()}."))
        return
    topologies = sorted({topology(run.row) for run in latency_runs})
    for topo in topologies:
        selected = [run for run in latency_runs if topology(run.row) == topo]
        label_by_shape = {
            (
                int(required_number(run.row, "capacity")),
                int(required_number(run.row, "payload")),
            ): workload_label(run.row)
            for run in selected
        }
        labels = [label_by_shape[key] for key in sorted(label_by_shape)]
        fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True, sharey=True)
        for ax, label in zip(axes.flat, labels):
            matching_runs = [
                run for run in selected if workload_label(run.row) == label
            ]
            for name in implementations:
                grouped = group_by(
                    (
                        run
                        for run in matching_runs
                        if run.row.get("implementation") == name
                    ),
                    lambda run: (run.row.get("fraction"),),
                )
                points = []
                for (fraction,), repeats in grouped.items():
                    x = number(repeats[0].row, "fraction")
                    ys = [
                        value
                        for run in repeats
                        if (value := metric(run)) is not None
                        and math.isfinite(value)
                        and value > 0
                    ]
                    if x is not None and ys:
                        points.append(
                            (x * 100, statistics.median(ys), min(ys), max(ys))
                        )
                points.sort()
                if not points:
                    continue
                xs, medians, lows, highs = zip(*points)
                ax.plot(
                    xs,
                    medians,
                    color=COLORS[name],
                    marker=MARKERS[name],
                    lw=1.8,
                    markersize=5,
                    label=name,
                )
                if any(low != high for low, high in zip(lows, highs)):
                    ax.fill_between(xs, lows, highs, color=COLORS[name], alpha=0.10)
            if reference is not None:
                ax.axhline(reference, color="#666666", linestyle="--", lw=1)
            ax.set(title=label, xlim=(47, 98), xticks=(50, 60, 70, 80, 90, 95))
            if log_axis:
                ax.set_yscale("log")
        for ax in axes.flat[len(labels) :]:
            ax.set_visible(False)
        fig.suptitle(f"{heading} · {topo}", y=0.99)
        fig.supxlabel("Offered load / slowest saturated median (%)")
        fig.supylabel(ylabel)
        fig.legend(
            handles=legend_handles(),
            loc="upper center",
            bbox_to_anchor=(0.5, 0.955),
            ncol=len(implementations),
            frameon=False,
        )
        fig.tight_layout(rect=(0, 0, 1, 0.90))
        plt.show()


workload_panels(
    lambda run: run.worst("p99_ns") / 1_000,
    "Worst consumer p99 (µs, log scale)",
    "Tail latency",
    log_axis=True,
)
if latency_runs:
    tails = [run.worst("p99_ns") / 1_000 for run in latency_runs]
    display(
        Markdown(
            f"**Readout:** The observed worst-consumer p99 summaries span "
            f"{min(tails):.2f}–{max(tails):.2f} µs across complete repeats. "
            "A rising curve indicates queueing/backpressure at higher offered load; "
            "inspect the achieved-rate section to see whether the rate was sustained."
        )
    )


## 4. Did the run sustain its offered rate?

**Question:** Did completed events keep pace with the scheduled arrivals? The plot shows achieved unique-event rate divided by offered rate; the dashed line is 100%. Each consumer row describes the same run, so it is counted once. This ratio diagnoses overload or timing effects; it is not a delivery-rate ratio.


In [ ]:
def attainment(run: LatencyRun) -> float | None:
    """Return achieved percent when both rate counters exist."""
    achieved = number(run.row, "events_per_second")
    offered = number(run.row, "offered_rate")
    return (
        achieved / offered * 100
        if achieved is not None and offered is not None and offered > 0
        else None
    )


workload_panels(
    attainment, "Achieved / offered rate (%)", "Rate attainment", reference=100
)
ratios = [value / 100 for run in latency_runs if (value := attainment(run)) is not None]
missing_rates = len(latency_runs) - len(ratios)
if ratios:
    below = sum(ratio < 0.95 for ratio in ratios)
    display(
        Markdown(
            f"**Readout:** {below}/{len(ratios)} complete latency repeats "
            f"achieved under 95% of their offered rate; {missing_rates} omit rate counters. "
            "Their tail values describe "
            "observed service under that shortfall, not successful delivery at the "
            "nominal schedule."
        )
    )
elif latency_runs:
    display(
        Markdown(
            f"No achieved/offered ratio is available for the {len(latency_runs)} "
            "complete latency repeats because rate counters are absent."
        )
    )


## 5. Saturation resource costs

**Question:** What CPU and process memory accompanied the saturated rates? Markers show medians and horizontal whiskers span the smallest to largest valid repeat. CPU cores are CPU time divided by elapsed time, so values above one mean multiple cores were busy. RSS is peak process resident memory, including runtime overhead; its axis is logarithmic. Missing counters are omitted, never converted to zero. Allocation and GC counters are intentionally left out because they do not describe equivalent work across runtimes.


In [ ]:
horizontal_panels(saturation, "cpu_cores", 1.0, "CPU cores used", "CPU at saturation")
horizontal_panels(
    saturation,
    "peak_rss_bytes",
    1024**2,
    "Peak process RSS (MiB, log scale)",
    "RSS at saturation",
    log_axis=True,
)
for field, scale, label in (
    ("cpu_cores", 1, "CPU cores"),
    ("peak_rss_bytes", 1024**2, "peak RSS MiB"),
):
    observed = [
        required_number(row, field) / scale
        for row in saturation
        if (number(row, field) or 0) > 0
    ]
    if observed:
        display(
            Markdown(
                f"**{label} readout:** {len(observed)}/{len(saturation)} valid "
                f"saturation repeats report this counter; the observed range is "
                f"{min(observed):.2f}–{max(observed):.2f}. Interpret it alongside "
                "the throughput and workload panels."
            )
        )


## 6. Workload detail: median and tails by consumer

**Question:** Is a broadcast consumer slower than its peers, and does that change with load? Set `DETAIL_WORKLOAD` to a workload key from the coverage table. The default selects the first measured broadcast workload, then the first measured workload if no broadcast data exist. Each line represents one implementation and consumer; its points are medians across valid repeats. Panels show p50, p99, and p99.9 separately, in microseconds on logarithmic axes. These are summaries of each consumer, not pooled event distributions.


In [ ]:
DETAIL_WORKLOAD: str | None = None  # e.g. "1p3c-2048-8b"
available = sorted({run.row["workload"] for run in latency_runs})
broadcast = [
    key
    for key in available
    if any(
        run.row["workload"] == key and (integer(run.row, "consumer_count") or 0) > 1
        for run in latency_runs
    )
]
selected_workload = DETAIL_WORKLOAD or (
    broadcast[0] if broadcast else (available[0] if available else None)
)
if selected_workload is None:
    display(Markdown("No complete latency runs are available for a detail plot."))
elif selected_workload not in available:
    display(
        Markdown(
            f"`{selected_workload}` has no complete latency repeats. "
            f"Available: {', '.join(available)}"
        )
    )
else:
    selected = [run for run in latency_runs if run.row["workload"] == selected_workload]
    fig, axes = plt.subplots(1, 3, figsize=(14, 5.8), sharex=True)
    percentiles = (("p50_ns", "p50"), ("p99_ns", "p99"), ("p999_ns", "p99.9"))
    for ax, (field, title) in zip(axes, percentiles):
        for name in implementations:
            consumer_ids = {
                int(required_number(row, "consumer"))
                for run in selected
                if run.row.get("implementation") == name
                for row in run.consumers
            }
            for consumer in sorted(consumer_ids):
                points = []
                fractions = {
                    required_number(run.row, "fraction")
                    for run in selected
                    if run.row.get("implementation") == name
                }
                for fraction in sorted(fractions):
                    values = [
                        required_number(row, field) / 1_000
                        for run in selected
                        if run.row.get("implementation") == name
                        and number(run.row, "fraction") == fraction
                        for row in run.consumers
                        if integer(row, "consumer") == consumer
                    ]
                    values = [value for value in values if value > 0]
                    if values:
                        points.append((fraction * 100, statistics.median(values)))
                if points:
                    xs, ys = zip(*points)
                    ax.plot(
                        xs,
                        ys,
                        color=COLORS[name],
                        marker=MARKERS[name],
                        linestyle=("-", "--", ":", "-.")[consumer % 4],
                        label=f"{name} / consumer {consumer}",
                    )
        ax.set(
            title=title,
            xlabel="Offered load (%)",
            ylabel="Latency (µs, log scale)",
            xlim=(47, 98),
            xticks=(50, 60, 70, 80, 90, 95),
        )
        ax.set_yscale("log")
    handles, labels = axes[0].get_legend_handles_labels()
    fig.suptitle(f"Consumer latency · {selected_workload}", y=0.98)
    fig.legend(
        handles,
        labels,
        loc="upper center",
        bbox_to_anchor=(0.5, 0.93),
        ncol=min(3, len(labels)),
        frameon=False,
    )
    fig.tight_layout(rect=(0, 0, 1, 0.77))
    plt.show()
    n_consumers = max(integer(run.row, "consumer_count") or 0 for run in selected)
    display(
        Markdown(
            f"**Readout:** `{selected_workload}` has {len(selected)} complete "
            f"repeats across the selected loads and implementations, with "
            f"{n_consumers} consumer(s) per repeat. Separate consumer curves "
            "show where broadcast tails diverge; compare points only at shared "
            "offered-load fractions."
        )
    )
